# C.2 — Đếm tham số và phép tính của một khối Transformer
**Đề tài 9, đề ngày 12/11/2026, tr. 19.** Phân tích đúng [`DecoderBlock` của A.1](../src/attention.py): self-attention nhân quả, hai LayerNorm pre-LN và FFN hai lớp với ReLU; không có cross-attention. [Vaswani et al., §3.2–3.3 và Bảng 1](../references/Attention.pdf) là tham chiếu cho kiến trúc và độ phức tạp.

Xét **một lần forward của một khối trên một chuỗi** $X\in\mathbb R^{n\times d}$, với $d=d_{\mathrm{model}}$, $f=d_{\mathrm{ff}}$, $h$ đầu, $h\mid d$ và $d_h=d/h$. Không tính embedding, PE hình sin, LayerNorm cuối mô hình, đầu đọc từ vựng, backward hay optimizer. Causal mask là buffer cố định, không phải tham số.

## 1. Số tham số
`Linear(a,b)` học $ab$ trọng số và $b$ bias. LayerNorm trên $d$ chiều học cả hệ số nhân $\gamma$ và bias $\beta$, mỗi vector dài $d$. Phép reshape/chia head chỉ tổ chức lại cùng $d$ chiều, không nhân số tham số lên $h$ lần.

| Thành phần | Trọng số hoặc scale | Bias | Tổng |
|---|---:|---:|---:|
| Chiếu Q, K, V gộp ($d\to 3d$) | $3d^2$ | $3d$ | $3d^2+3d$ |
| Chiếu đầu ra ($d\to d$) | $d^2$ | $d$ | $d^2+d$ |
| FFN — lớp 1 ($d\to f$) | $df$ | $f$ | $df+f$ |
| FFN — lớp 2 ($f\to d$) | $df$ | $d$ | $df+d$ |
| Hai LayerNorm | $2d$ | $2d$ | $4d$ |

$$N_{\mathrm{params}}=4d^2+2df+9d+f.$$

Với $d,f$ cố định, số tham số không phụ thuộc $n$ hay $h$. ReLU, softmax, phép cộng residual và causal mask không có tham số học.

## 2. Số phép tính forward
Quy ước **MAC**: tích ma trận $(a\times b)(b\times c)$ dùng $abc$ lượt nhân–tích lũy. Khi đối chiếu với profiler, dùng **1 MAC = 2 FLOP** theo quy ước GEMM (một nhân và một cộng); đây không phải đếm từng lệnh máy hoặc cộng bias. Bảng chỉ tính MAC của các tích ma trận.

| Thành phần | Kích thước tích | MAC trên cả khối |
|---|---|---:|
| Chiếu Q, K, V | $(n\times d)(d\times 3d)$ | $3nd^2$ |
| Tính điểm $QK^{\mathsf T}$ | Mỗi head: $(n\times d_h)(d_h\times n)$ | $hn^2d_h=n^2d$ |
| Tổng có trọng số $PV$ | Mỗi head: $(n\times n)(n\times d_h)$, $P$ là xác suất sau softmax | $hn^2d_h=n^2d$ |
| Chiếu đầu ra | $(n\times d)(d\times d)$ | $nd^2$ |
| FFN, hai lớp | $(n\times d)(d\times f)$ và $(n\times f)(f\times d)$ | $2ndf$ |

$$\mathrm{MAC}_{\mathrm{matmul}}=4nd^2+2ndf+2n^2d,\qquad \mathrm{FLOP}_{\mathrm{matmul}}=2\,\mathrm{MAC}_{\mathrm{matmul}}.$$

**Ngoài các tích ma trận:** cộng bias tuyến tính $n(5d+f)$ phép cộng; hai residual $2nd$ phép cộng; ReLU $\Theta(nf)$, hai LayerNorm $\Theta(nd)$; scale điểm $hn^2$ phép nhân và softmax xử lý $hn^2$ phần tử, chi phí $\Theta(hn^2)$. Gán mask có chi phí xử lý $O(hn^2)$ phần tử, không tính thành FLOP. Với $h,d,f$ cố định, chính **hai tích $QK^{\mathsf T}$ và $PV$**, cùng các thao tác trên ma trận điểm, tạo hạng tử bậc hai theo $n$; phần projection và FFN tăng tuyến tính theo $n$. Vì $h\le d$, độ phức tạp toàn khối là $\Theta(nd^2+ndf+n^2d)$; score và xác suất attention có kích thước $h\times n\times n$. Với batch $B$, số phép tính forward nhân $B$, số tham số giữ nguyên.

**Vì sao không chia đôi khi bật causal mask?** A.1 tính đủ ma trận $n\times n$ rồi mới điền $-\infty$ phía trên đường chéo. Phép nhân dense $PV$ vẫn chạy với tensor đủ kích thước, dù các trọng số bị che bằng 0. Chỉ thuật toán thực sự bỏ qua ô bị che mới tận dụng được $n(n+1)/2$ cặp hợp lệ; mask trong mã A.1 không làm việc đó.

## 3. Đối chiếu trên CPU với mã A.1
Dùng $d=64$, $f=128$, $h=4$, $n=128$ từ cấu hình A.2; chỉ khởi tạo **một block**, không tải checkpoint hay huấn luyện. Bảng phép tính thay đổi $n=64,128,256$ để thấy khi $n$ gấp đôi thì chiếu/FFN gấp đôi, còn hai tích attention gấp bốn. Hàng $n=256$ chỉ là kiểm chứng C.2, không phải độ dài đã huấn luyện ở A.2. Hàng $h=8$ giữ nguyên $d$ để kiểm tra sự triệt tiêu của $h\,d_h$.

Chạy tuần tự từ thư mục gốc repo (hoặc `notebooks/`) với PyTorch. `numel()` đếm số phần tử thực của các tham số học. [PyTorch Profiler](https://docs.pytorch.org/docs/stable/profiler.html) ước lượng FLOP từ các phép nhân ma trận thực sự xuất hiện trong forward CPU: `aten::addmm` cho chiếu/FFN, `aten::bmm` cho hai tích attention. Không dùng thời gian chạy để suy ra FLOP và không gọi đây là phép đo toàn bộ forward.

In [ ]:
from pathlib import Path
import platform
import sys

import torch
from torch.profiler import ProfilerActivity, profile

here = Path.cwd().resolve()
root = next((p for p in (here, *here.parents) if (p / 'src/attention.py').is_file()), None)
if root is None:
    raise RuntimeError('Mở notebook từ thư mục gốc repo hoặc thư mục notebooks/.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.attention import DecoderBlock

print(f'Python {platform.python_version()} | PyTorch {torch.__version__} | CPU')
d, f, n_a2 = 64, 128, 128
blocks = {h: DecoderBlock(d, h, f, max_len=256).eval() for h in (4, 8)}
expected = {
    'attn.qkv': 3*d*d + 3*d,
    'attn.out_proj': d*d + d,
    'ffn.0': d*f + f,
    'ffn.2': f*d + d,
    'norm1': 2*d,
    'norm2': 2*d,
}
total = sum(expected.values())
for h, block in blocks.items():
    actual = {
        name: sum(p.numel() for p in block.get_submodule(name).parameters() if p.requires_grad)
        for name in expected
    }
    assert actual == expected, (h, actual, expected)
    assert sum(p.numel() for p in block.parameters() if p.requires_grad) == total
    if h == 4:
        for name, count in actual.items():
            print(f'{name:>14}: {count:,}')
    print(f'{h} head: {total:,} tham số học')

In [ ]:
def matmul_flops(block, n):
    x = torch.zeros(1, n, d, device='cpu')
    with torch.inference_mode(), profile(
        activities=[ProfilerActivity.CPU], record_shapes=True, with_flops=True,
    ) as trace:
        block(x)
    events = {event.key: event for event in trace.key_averages()}
    return events['aten::addmm'].flops, events['aten::bmm'].flops

print('n    head   MAC chiếu + FFN  MAC attention   MAC tổng      FLOP matmul (profiler)')
for n, h in ((64, 4), (n_a2, 4), (256, 4), (n_a2, 8)):
    linear = 4*n*d*d + 2*n*d*f
    attention = 2*n*n*d
    measured_linear, measured_attention = matmul_flops(blocks[h], n)
    assert measured_linear == 2*linear, (n, h, measured_linear, 2*linear)
    assert measured_attention == 2*attention, (n, h, measured_attention, 2*attention)
    print(f'{n:3} {h:6} {linear:17,} {attention:14,} {linear+attention:13,} '
          f'{measured_linear+measured_attention:23,}')

**Kết luận cho C.2.** Với cấu hình A.2, một khối ở $n=128$ có **33.472 tham số** và **6.291.456 MAC** cho các tích ma trận (**12.582.912 FLOP** theo quy ước trên). Hai tích attention chiếm đúng $1/3$ MAC ở cấu hình này; tỉ lệ đó tăng khi chuỗi dài hơn. Không dùng số FLOP của một khối forward làm FLOP huấn luyện toàn mô hình.